# Meta FAISS with LCEL Pipelines - Part 2 (Lecture 33)

This notebook integrates Meta FAISS into a modern LangChain Expression Language (LCEL) RAG pipeline.
It demonstrates:
1. Setting up a FAISS retriever with customized search parameters.
2. Building an end-to-end LCEL chain with Groq LLM.
3. Streaming responses token-by-token.
4. Comprehensive assertion testing.

## 1. Setup & Vector Store Preparation
Initialize FAISS in-memory with technical documentation.

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

corpus = [
    Document(page_content="FAISS was developed by Meta AI to provide billion-scale similarity search on GPUs and CPUs.", metadata={"source": "meta_faiss.pdf"}),
    Document(page_content="Inverted File Index (IVF) partitions vector space into Voronoi cells to accelerate retrieval.", metadata={"source": "faiss_indexing.pdf"}),
    Document(page_content="Hierarchical Navigable Small World (HNSW) graphs offer exceptional recall-latency trade-offs for approximate nearest neighbors.", metadata={"source": "hnsw_guide.pdf"}),
]

vector_store = FAISS.from_documents(corpus, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print(f"FAISS index contains {vector_store.index.ntotal} vectors.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_4656\1725983048.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index contains 3 vectors.


## 2. LCEL Chain Construction
Assemble the pipeline with ChatGroq.

In [2]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source')}]: {d.page_content}" for d in docs)

template = """You are a database engineer. Answer the question using only the provided context:
Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("FAISS LCEL chain assembled.")

FAISS LCEL chain assembled.


## 3. Streaming Query & Assertion Verification
Stream the response and verify retrieval accuracy.

In [3]:
query = "What graph structure provides great recall-latency trade-offs in ANN?"
print(f"Query: {query}\n")

streamed_chunks = []
for chunk in rag_chain.stream(query):
    streamed_chunks.append(chunk)
    print(chunk, end="", flush=True)
print("\n")

full_response = "".join(streamed_chunks)
assert "hnsw" in full_response.lower() or "hierarchical" in full_response.lower(), "Response must identify HNSW"
print("FAISS LCEL streaming test passed.")

Query: What graph structure provides great recall-latency trade-offs in ANN?



Hier

archical

 Navig

able

 Small

... (6 outputs left out)

)

 graphs

.



FAISS LCEL streaming test passed.
